# Sesión 5 · APIs: pedir datos en vez de rasparlos

En la sesión anterior sacamos datos de una página web. Eso siempre es frágil:
si cambian el diseño, el código se rompe.

Una **API** es una puerta que la institución abre a propósito para que le
pidas datos. Es más estable y devuelve todo ya ordenado.

| | Scraping | API |
|---|---|---|
| Qué devuelve | HTML para humanos | JSON para programas |
| Estabilidad | Se rompe si cambia el diseño | Estable |
| Permiso | Zona gris | Explícito |

La segunda mitad de la clase usa una API distinta: una que entiende texto.

In [1]:
import pandas as pd
import requests

## 1. Anatomía de una consulta

Toda API tiene una **URL base** (el endpoint) y **parámetros** que afinan
el pedido. Vamos por la población del Perú al Banco Mundial.

In [2]:
url = "https://api.worldbank.org/v2/country/PER/indicator/SP.POP.TOTL"

parametros = {
    "format": "json",      # sin esto devuelve XML
    "date": "2015:2021",   # rango de años
    "per_page": 100,
}

respuesta = requests.get(url, params=parametros, timeout=30)
respuesta.status_code

200

### La URL que se armó

`params` construye la URL con el formato correcto. Es más seguro que pegar
texto a mano.

In [3]:
respuesta.url

'https://api.worldbank.org/v2/country/PER/indicator/SP.POP.TOTL?format=json&date=2015%3A2021&per_page=100'

## 2. El JSON que llega

`.json()` convierte la respuesta en listas y diccionarios de Python.

In [4]:
datos = respuesta.json()
type(datos), len(datos)

(list, 2)

Esta API devuelve una lista de dos elementos: primero los **metadatos**,
después los **datos**. Cada API tiene su forma; siempre hay que mirar.

In [5]:
datos[0]

{'page': 1,
 'pages': 1,
 'per_page': 100,
 'total': 7,
 'sourceid': '2',
 'lastupdated': '2026-07-13'}

### Una observación individual

Fíjate que está **anidada**: dentro de `country` e `indicator` hay más
diccionarios.

In [6]:
datos[1][0]

{'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'},
 'country': {'id': 'PE', 'value': 'Peru'},
 'countryiso3code': 'PER',
 'date': '2021',
 'value': 33155882,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

## 3. De JSON a DataFrame

`pd.DataFrame` lee una lista de diccionarios directamente.

In [7]:
poblacion = pd.DataFrame(datos[1])
poblacion.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'PE', 'value': 'Peru'}",PER,2021,33155882,,,0
1,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'PE', 'value': 'Peru'}",PER,2020,32838579,,,0
2,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'PE', 'value': 'Peru'}",PER,2019,32449303,,,0
3,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'PE', 'value': 'Peru'}",PER,2018,31897584,,,0
4,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'PE', 'value': 'Peru'}",PER,2017,31324637,,,0


### Aplanar lo que quedó anidado

La columna `country` tiene un diccionario adentro. Se saca con `.apply`.

In [8]:
poblacion["pais"] = poblacion["country"].apply(lambda d: d["value"])
poblacion = poblacion[["pais", "date", "value"]].rename(
    columns={"date": "anio", "value": "poblacion"}
)
poblacion["anio"] = poblacion["anio"].astype(int)
poblacion = poblacion.sort_values("anio")
poblacion

,pais,anio,poblacion
6,Peru,2015,30457600
5,Peru,2016,30866494
4,Peru,2017,31324637
3,Peru,2018,31897584
2,Peru,2019,32449303
1,Peru,2020,32838579
0,Peru,2021,33155882


## 4. Consultar varios indicadores con una función

Cuando se repite el mismo pedido cambiando un dato, se escribe una función.
Esta es la única función que necesitamos en todo el curso.

In [9]:
def pedir_indicador(codigo, pais="PER", desde=2015, hasta=2021):
    """Trae un indicador del Banco Mundial como DataFrame."""
    url = f"https://api.worldbank.org/v2/country/{pais}/indicator/{codigo}"
    r = requests.get(
        url,
        params={"format": "json", "date": f"{desde}:{hasta}", "per_page": 100},
        timeout=30,
    )
    r.raise_for_status()          # si algo falló, avisa aquí y no más adelante
    filas = r.json()[1]
    df = pd.DataFrame(filas)[["date", "value"]]
    df.columns = ["anio", codigo]
    df["anio"] = df["anio"].astype(int)
    return df.sort_values("anio")


pedir_indicador("SP.POP.TOTL")

,anio,SP.POP.TOTL
6,2015,30457600
5,2016,30866494
4,2017,31324637
3,2018,31897584
2,2019,32449303
1,2020,32838579
0,2021,33155882


### Ahora es barato pedir más cosas

- `SH.XPD.CHEX.PC.CD` — gasto en salud por persona (US$)
- `SP.RUR.TOTL.ZS` — población rural (% del total)

In [10]:
gasto = pedir_indicador("SH.XPD.CHEX.PC.CD")
rural = pedir_indicador("SP.RUR.TOTL.ZS")

contexto = (
    pedir_indicador("SP.POP.TOTL")
    .merge(gasto, on="anio")
    .merge(rural, on="anio")
)
contexto.columns = ["anio", "poblacion", "gasto_salud_pc", "pct_rural"]
contexto

,anio,poblacion,gasto_salud_pc,pct_rural
0,2015,30457600,310.609833,19.301722
1,2016,30866494,311.335175,18.580764
2,2017,31324637,332.543213,17.806835
3,2018,31897584,362.759949,17.314507
4,2019,32449303,367.987396,16.896700
5,2020,32838579,399.138000,16.473256
6,2021,33155882,455.356079,16.051988


## 5. Cuando algo sale mal

Una API puede fallar. Se maneja con `try`, no se ignora.

In [11]:
try:
    r = requests.get("https://api.worldbank.org/v2/esta-ruta-no-existe",
                     params={"format": "json"}, timeout=10)
    r.raise_for_status()
    print(r.json())
except requests.exceptions.RequestException as e:
    print("Falló la consulta:", type(e).__name__)

Falló la consulta: HTTPError


### Cuidado: no todo error viene con código de error

Algunas APIs responden `200` aunque el pedido esté mal, y meten el problema
dentro del JSON. Por eso no basta con `raise_for_status`: hay que **mirar lo
que llegó**.

In [12]:
r = requests.get("https://api.worldbank.org/v2/country/XXX/indicator/NADA",
                 params={"format": "json"}, timeout=10)
print("Status:", r.status_code)     # dice 200...
r.json()                            # ...pero adentro viene el error

Status: 200


[{'message': [{'id': '120',
    'key': 'Invalid value',
    'value': 'The provided parameter value is not valid'},
   {'id': '120',
    'key': 'Invalid value',
    'value': 'The provided parameter value is not valid'}]}]

### Si vas a pedir muchas veces, espera entre pedidos

```python
import time
for codigo in lista_de_codigos:
    df = pedir_indicador(codigo)
    time.sleep(0.5)      # medio segundo entre consultas
```

Sin esto te devuelven `429 Too Many Requests` y te bloquean.

In [13]:
contexto.to_csv("contexto_peru.csv", index=False)
contexto.shape

(7, 4)

---
# Segunda parte · Una API que entiende texto

Todo lo anterior funciona porque los datos ya eran una tabla. Pero muchísima
información pública está en **texto libre**: reclamos, actas, expedientes,
respuestas abiertas de encuestas.

Antes eso se leía a mano. Hoy se le pide a un modelo de lenguaje, con
exactamente la misma mecánica: una llamada a una API.

In [14]:
reclamos = pd.read_csv("../../data/05-apis-e-ia/reclamos_salud.csv")
reclamos.head()

,id,texto,departamento
0,1,El puesto de salud de Castilla no tiene prueba...,Piura
1,2,Solicito fumigacion en el AAHH Nueva Esperanza...,Piura
2,3,La posta de Tarapoto atiende solo hasta las 2p...,San Martin
3,4,Reclamo por cobro indebido de 50 soles por con...,Loreto
4,5,Hace un mes reporte criaderos de zancudos fren...,Tumbes


In [15]:
print(reclamos.loc[0, "texto"])

El puesto de salud de Castilla no tiene pruebas de dengue desde hace tres semanas. Vine dos veces y me mandaron a Piura.


**La pregunta:** ¿de qué se queja la gente? Con 20 reclamos se puede leer.
Con 5,000 no.

## 6. Configurar la clave

La clave va en un archivo `.env`, **nunca** escrita dentro del notebook.
Si la escribes en el código y subes el notebook a GitHub, la clave queda
pública y cualquiera puede gastar con tu cuenta.

1. Copia `.env.example` a `.env`
2. Pega tu clave ahí
3. `.env` ya está en el `.gitignore`

In [16]:
import os
from dotenv import load_dotenv

load_dotenv()
tengo_clave = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Clave configurada:", tengo_clave)

Clave configurada: False


## 7. La primera llamada

Es lo mismo que el Banco Mundial: se manda un pedido, se recibe una respuesta.
La diferencia es que el pedido está escrito en español.

In [ ]:
from anthropic import Anthropic

cliente = Anthropic()

respuesta = cliente.messages.create(
    model="claude-sonnet-5",
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": "¿Qué es el dengue? Responde en dos oraciones.",
    }],
)
print(respuesta.content[0].text)

## 8. Clasificar un reclamo

La clave está en pedir **una respuesta corta y cerrada**. Si dejas que el
modelo escriba libre, no puedes ponerlo en una columna.

In [ ]:
CATEGORIAS = ["Desabastecimiento", "Infraestructura", "Trato al usuario",
              "Tiempos de espera", "Vectores y saneamiento", "Acceso geográfico",
              "Felicitación", "Otro"]

def clasificar(texto):
    """Devuelve una sola categoría de la lista."""
    r = cliente.messages.create(
        model="claude-sonnet-5",
        max_tokens=20,
        messages=[{
            "role": "user",
            "content": (
                f"Clasifica este reclamo de salud en UNA de estas categorías:\n"
                f"{', '.join(CATEGORIAS)}\n\n"
                f"Reclamo: {texto}\n\n"
                f"Responde solo con la categoría, sin explicar."
            ),
        }],
    )
    return r.content[0].text.strip()


clasificar(reclamos.loc[0, "texto"])

## 9. Aplicarlo a toda la tabla

Con `.apply` se recorre la columna. Cada fila es una llamada a la API, así
que esto **cuesta dinero y toma tiempo**: siempre probar primero con 3 filas.

In [ ]:
muestra = reclamos.head(3).copy()
muestra["categoria"] = muestra["texto"].apply(clasificar)
muestra[["id", "categoria", "texto"]]

### Si la muestra salió bien, ahora sí todo

In [ ]:
reclamos["categoria"] = reclamos["texto"].apply(clasificar)
reclamos["categoria"].value_counts()

## 10. Extraer varios campos a la vez

Se le pide que responda en JSON y se convierte a columnas. Así, de un texto
libre salen tres variables de golpe.

In [ ]:
import json

def extraer(texto):
    r = cliente.messages.create(
        model="claude-sonnet-5",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": (
                f"Extrae de este reclamo y responde SOLO con JSON válido, "
                f"sin texto adicional:\n"
                f'{{"categoria": una de {CATEGORIAS}, '
                f'"urgencia": "alta"|"media"|"baja", '
                f'"menciona_dengue": true|false}}\n\n'
                f"Reclamo: {texto}"
            ),
        }],
    )
    return json.loads(r.content[0].text.strip())


extraer(reclamos.loc[5, "texto"])

In [ ]:
campos = reclamos["texto"].apply(extraer).apply(pd.Series)
resultado = pd.concat([reclamos[["id", "departamento", "texto"]], campos], axis=1)
resultado.head()

## 11. Ahora ya es una tabla normal

Esto es lo importante: el texto libre se volvió datos, y a partir de aquí se
usa todo lo de la sesión 3.

In [ ]:
resultado.pivot_table(
    index="categoria", columns="urgencia", values="id", aggfunc="count", fill_value=0
)

In [ ]:
resultado.groupby("departamento")["menciona_dengue"].sum().sort_values(ascending=False)

In [ ]:
resultado.to_csv("reclamos_clasificados.csv", index=False)
resultado.shape

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Pedir a una API | `requests.get(url, params={...})` |
| Ver la URL armada | `.url` |
| JSON a Python | `.json()` |
| Fallar a tiempo | `.raise_for_status()` |
| Aplanar anidados | `.apply(lambda d: d["value"])` |
| Manejar errores | `try / except RequestException` |
| Clave secreta | `.env` + `load_dotenv()` |
| Clasificar texto | `cliente.messages.create(...)` |
| Aplicar a la tabla | `.apply(funcion)` |
| JSON a columnas | `.apply(pd.Series)` |

### Las tres reglas al usar un modelo sobre una tabla

1. **Prueba con 3 filas antes de correr 5,000.** Cada fila cuesta.
2. **Pide respuestas cerradas.** Una lista fija de categorías, o JSON.
3. **Revisa a mano una muestra del resultado.** El modelo se equivoca, y si
   no revisas, el error entra en tu informe con cara de dato.